In [1]:
import numpy as np
import math
import time

# ============================================================
# SU(2) as unit quaternions: U = q0 I + i (q·σ)
# ============================================================

def quat_mul(q, p):
    a0,a1,a2,a3 = q
    b0,b1,b2,b3 = p
    return np.array([
        a0*b0 - (a1*b1 + a2*b2 + a3*b3),
        a0*b1 + b0*a1 + (a2*b3 - a3*b2),
        a0*b2 + b0*a2 + (a3*b1 - a1*b3),
        a0*b3 + b0*a3 + (a1*b2 - a2*b1)
    ], dtype=float)

def quat_conj(q):
    return np.array([q[0], -q[1], -q[2], -q[3]], dtype=float)

def quat_norm(q):
    return float(np.linalg.norm(q))

def quat_normalize(q):
    n = quat_norm(q)
    if n < 1e-15:
        return np.array([1.0,0.0,0.0,0.0], dtype=float)
    return q / n

def su2_quat_from_x(x):
    """
    x in R^3 parameterizes U = exp(i (x·σ)/2).
    Return quaternion q and Haar Jacobian J(r) = (sin(r/2)/(r/2))^2.
    """
    x = np.asarray(x, dtype=float)
    r = float(np.linalg.norm(x))
    if r < 1e-14:
        return np.array([1.0,0.0,0.0,0.0], dtype=float), 1.0
    half = 0.5 * r
    c = math.cos(half)
    s = math.sin(half)
    s_over_r = s / r
    q = np.array([c, s_over_r*x[0], s_over_r*x[1], s_over_r*x[2]], dtype=float)
    J = (s / half)**2
    return q, J

def x_from_su2_quat(q):
    """
    Principal-branch log map: q=[cos(r/2), sin(r/2) n] -> x = r n.
    """
    q = np.asarray(q, dtype=float)
    q0 = float(q[0])
    v = q[1:]
    s = float(np.linalg.norm(v))
    if s < 1e-14:
        return np.zeros(3)
    half_r = math.atan2(s, q0)     # r/2 in [0, pi]
    r = 2.0 * half_r
    n = v / s
    return r * n

def quat_to_rot(q):
    """
    SO(3) adjoint rotation from SU(2) quaternion.
    """
    w,x,y,z = q
    ww, xx, yy, zz = w*w, x*x, y*y, z*z
    wx, wy, wz = w*x, w*y, w*z
    xy, xz, yz = x*y, x*z, y*z
    return np.array([
        [ww + xx - yy - zz, 2*(xy - wz),       2*(xz + wy)],
        [2*(xy + wz),       ww - xx + yy - zz, 2*(yz - wx)],
        [2*(xz - wy),       2*(yz + wx),       ww - xx - yy + zz]
    ], dtype=float)

def re_tr_from_quat(q):
    # Re Tr(U) = 2 q0
    return 2.0 * float(q[0])

def cross_matrix(v):
    x,y,z = v
    return np.array([[0.0, -z,  y],
                     [ z, 0.0, -x],
                     [-y,  x, 0.0]], dtype=float)

# ============================================================
# 2x2 open block geometry (9 vertices, 12 links, 4 plaquettes)
# ============================================================

def build_2x2_open_block():
    vid = {(i,j): j*3 + i for j in range(3) for i in range(3)}  # 0..8
    edges = []
    # horizontals: (i,j)->(i+1,j), i=0..1, j=0..2
    for j in range(3):
        for i in range(2):
            edges.append((vid[(i,j)], vid[(i+1,j)], ('h',i,j)))
    # verticals: (i,j)->(i,j+1), i=0..2, j=0..1
    for j in range(2):
        for i in range(3):
            edges.append((vid[(i,j)], vid[(i,j+1)], ('v',i,j)))

    eidx = {tag:k for k,(_,_,tag) in enumerate(edges)}

    plaquettes = []
    for j in range(2):
        for i in range(2):
            e_r = eidx[('h', i, j)]
            e_u = eidx[('v', i+1, j)]
            e_l = eidx[('h', i, j+1)]
            e_d = eidx[('v', i, j)]
            plaquettes.append((e_r,e_u,e_l,e_d))

    return 9, edges, plaquettes

Vn, edges, plaquettes = build_2x2_open_block()

# ============================================================
# Gauge fixing by maximizing F[g] = sum ReTr(g_t U_e g_h^dagger)
# ============================================================

def gauge_functional(U_links, edges, g_vertices):
    F = 0.0
    for e,(tail,head,_) in enumerate(edges):
        q = quat_mul(g_vertices[tail], U_links[e])
        q = quat_mul(q, quat_conj(g_vertices[head]))
        F += re_tr_from_quat(q)
    return F

def gauge_fix_landau(U_links, edges, root=0, max_sweeps=500, tol=1e-10, omega_over=1.7):
    """
    Local SU(2) relaxation/overrelaxation to maximize F.
    Pin g_root = I.
    """
    Vn = max(max(t,h) for t,h,_ in edges) + 1
    g = [np.array([1.0,0.0,0.0,0.0], dtype=float) for _ in range(Vn)]

    outgoing = [[] for _ in range(Vn)]
    incoming = [[] for _ in range(Vn)]
    for e,(t,h,_) in enumerate(edges):
        outgoing[t].append(e)
        incoming[h].append(e)

    def quat_pow(q, alpha):
        q = quat_normalize(q)
        w = float(q[0])
        v = q[1:]
        s = float(np.linalg.norm(v))
        if s < 1e-14:
            return np.array([1.0,0.0,0.0,0.0], dtype=float)
        theta = math.atan2(s, w)
        n = v / s
        theta_a = alpha * theta
        return np.array([math.cos(theta_a), *(math.sin(theta_a)*n)], dtype=float)

    for sweep in range(max_sweeps):
        max_change = 0.0
        for v in range(Vn):
            if v == root:
                continue
            hq = np.zeros(4, dtype=float)

            # outgoing edges: U_e g_head^dagger
            for e in outgoing[v]:
                _, head, _ = edges[e]
                hq += quat_mul(U_links[e], quat_conj(g[head]))

            # incoming edges: U_e^dagger g_tail^dagger
            for e in incoming[v]:
                tail, _, _ = edges[e]
                hq += quat_mul(quat_conj(U_links[e]), quat_conj(g[tail]))

            g_opt = quat_normalize(quat_conj(hq)) if quat_norm(hq) > 1e-14 else np.array([1.0,0.0,0.0,0.0])

            # overrelax step: g_new = (g_opt g_old^{-1})^{omega} g_old
            delta = quat_mul(g_opt, quat_conj(g[v]))
            g_new = quat_mul(quat_pow(delta, omega_over), g[v])
            g_new = quat_normalize(g_new)

            ch = float(np.linalg.norm(g_new - g[v]))
            max_change = max(max_change, ch)
            g[v] = g_new

        if max_change < tol:
            break

    # gauge-fixed links
    U_g = []
    for e,(t,h,_) in enumerate(edges):
        q = quat_mul(g[t], U_links[e])
        q = quat_mul(q, quat_conj(g[h]))
        U_g.append(quat_normalize(q))
    return g, np.array(U_g)

def gauge_fix_X_to_slice(X_flat, root=0):
    X_flat = np.asarray(X_flat, dtype=float)
    E = len(edges)
    X = X_flat.reshape(E,3)
    U = np.empty((E,4), dtype=float)
    for e in range(E):
        U[e], _ = su2_quat_from_x(X[e])
    g, U_g = gauge_fix_landau(U, edges, root=root)
    X_g = np.zeros((E,3), dtype=float)
    for e in range(E):
        X_g[e] = x_from_su2_quat(U_g[e])
    return X_g.reshape(-1), U_g

# ============================================================
# Symmetric FP matrix: M = -Hess_g F at the gauge-fixed point
# ============================================================

def fp_matrix_from_links(U_links, edges, root=0):
    Vn = max(max(t,h) for t,h,_ in edges)+1
    dim = 3*Vn
    M = np.zeros((dim, dim), dtype=float)
    I3 = np.eye(3)

    for e,(t,h,_) in enumerate(edges):
        q = U_links[e]
        q0 = float(q[0])
        qv = q[1:]

        w = 0.5*q0                 # (ReTr(U))/4 = q0/2
        K = w*I3 + 0.5*cross_matrix(qv)

        M[3*t:3*t+3, 3*t:3*t+3] += w*I3
        M[3*h:3*h+3, 3*h:3*h+3] += w*I3
        M[3*t:3*t+3, 3*h:3*h+3] += -K
        M[3*h:3*h+3, 3*t:3*t+3] += -K.T

    # remove root block
    idx = np.arange(dim)
    rm = np.arange(3*root, 3*root+3)
    keep = np.setdiff1d(idx, rm)
    M_red = M[np.ix_(keep, keep)]
    return 0.5*(M_red + M_red.T)

# ============================================================
# Wilson + Haar + FP potential
# ============================================================

def wilson_action_from_quats(U_quats, plaquettes):
    S = 0.0
    for (e_r, e_u, e_l, e_d) in plaquettes:
        q = quat_mul(U_quats[e_r], U_quats[e_u])
        q = quat_mul(q, quat_conj(U_quats[e_l]))
        q = quat_mul(q, quat_conj(U_quats[e_d]))
        S += 1.0 - float(q[0])     # 1 - (0.5 ReTr) = 1 - q0
    return S

def potential_V(X_flat, beta=1.0, root=0):
    X_flat = np.asarray(X_flat, dtype=float)
    E = len(edges)
    X = X_flat.reshape(E,3)

    U = np.empty((E,4), dtype=float)
    logJ_sum = 0.0
    for e in range(E):
        U[e], J = su2_quat_from_x(X[e])
        logJ_sum += math.log(J)

    S = wilson_action_from_quats(U, plaquettes)
    Mred = fp_matrix_from_links(U, edges, root=root)
    sign, logdet = np.linalg.slogdet(Mred)
    if sign <= 0 or not np.isfinite(logdet):
        return float("inf")
    return beta*S - logJ_sum - logdet

# ============================================================
# H/V decomposition from covariant orbit map D(U)
# ============================================================

def build_covariant_D(U_quats, edges, root=0):
    Vn = max(max(t,h) for t,h,_ in edges)+1
    E = len(edges)
    d=3
    v_map = {}
    k=0
    for v in range(Vn):
        if v==root:
            continue
        v_map[v]=k
        k += 1

    D = np.zeros((d*E, d*(Vn-1)), dtype=float)
    I3 = np.eye(3)
    for e,(t,h,_) in enumerate(edges):
        R = quat_to_rot(U_quats[e])
        r0 = 3*e
        if t != root:
            c0 = 3*v_map[t]
            D[r0:r0+3, c0:c0+3] += I3
        if h != root:
            c1 = 3*v_map[h]
            D[r0:r0+3, c1:c1+3] += -R
    return D

def bases_from_D(D, tol=1e-10):
    U,S,VT = np.linalg.svd(D, full_matrices=True)
    r = int(np.sum(S>tol))
    Vbasis = U[:, :r]
    Hbasis = U[:, r:]
    return Vbasis, Hbasis

# ============================================================
# Finite-difference Hessian and constants
# ============================================================

def hessian_fd(f, x, eps=2e-5):
    x = np.asarray(x, dtype=float)
    n = x.size
    H = np.zeros((n,n), dtype=float)
    fx = float(f(x))

    f_plus = np.zeros(n)
    f_minus = np.zeros(n)

    for i in range(n):
        ei = np.zeros(n); ei[i]=1.0
        f_plus[i] = float(f(x + eps*ei))
        f_minus[i] = float(f(x - eps*ei))
        H[i,i] = (f_plus[i] - 2.0*fx + f_minus[i])/(eps*eps)

    for i in range(n):
        ei = np.zeros(n); ei[i]=1.0
        for j in range(i+1,n):
            ej = np.zeros(n); ej[j]=1.0
            fpp = float(f(x + eps*ei + eps*ej))
            fpm = float(f(x + eps*ei - eps*ej))
            fmp = float(f(x - eps*ei + eps*ej))
            fmm = float(f(x - eps*ei - eps*ej))
            Hij = (fpp - fpm - fmp + fmm)/(4.0*eps*eps)
            H[i,j]=Hij
            H[j,i]=Hij

    return H

def constants_at_point(X_flat, beta=1.0, eps=2e-5, root=0):
    # build U at point for D basis
    E=len(edges)
    X = np.asarray(X_flat).reshape(E,3)
    U = np.empty((E,4), dtype=float)
    for e in range(E):
        U[e], _ = su2_quat_from_x(X[e])

    D = build_covariant_D(U, edges, root=root)
    Vbasis, Hbasis = bases_from_D(D)

    f = lambda z: potential_V(z, beta=beta, root=root)
    H = hessian_fd(f, np.asarray(X_flat), eps=eps)

    HHH = 0.5*(Hbasis.T @ H @ Hbasis + (Hbasis.T @ H @ Hbasis).T)
    HVV = 0.5*(Vbasis.T @ H @ Vbasis + (Vbasis.T @ H @ Vbasis).T)
    HHV = Hbasis.T @ H @ Vbasis

    kH = float(np.min(np.linalg.eigvalsh(HHH)))
    kV = float(np.min(np.linalg.eigvalsh(HVV)))
    eta = float(np.linalg.norm(HHV, 2))

    kBE_full  = min(kV, kH - eta*eta/kV) if kV>0 else float("nan")
    kBE_slice = (kH - eta*eta/kV) if kV>0 else float("nan")

    # FP minimum eigenvalue (reference)
    fp_min = float(np.min(np.linalg.eigvalsh(fp_matrix_from_links(U, edges, root=root))))

    return kH, kV, eta, kBE_full, kBE_slice, fp_min

def random_ball(rng, n, R):
    v = rng.normal(size=n)
    v /= np.linalg.norm(v)
    r = (rng.random() ** (1.0/n)) * R
    return r * v

def sample_on_slice(beta=1.0, R=0.05, nsamp=10, eps=2e-5, seed=7, root=0):
    rng = np.random.default_rng(seed)

    kH_min = float("inf")
    kV_min = float("inf")
    eta_max = 0.0
    fp_min = float("inf")

    for _ in range(nsamp):
        X0 = random_ball(rng, 36, R)
        X_gf, U_gf = gauge_fix_X_to_slice(X0, root=root)

        kH, kV, eta, kBE_full, kBE_slice, fp = constants_at_point(X_gf, beta=beta, eps=eps, root=root)

        kH_min = min(kH_min, kH)
        kV_min = min(kV_min, kV)
        eta_max = max(eta_max, eta)
        fp_min = min(fp_min, fp)

    kBE_full  = min(kV_min, kH_min - eta_max*eta_max/kV_min) if kV_min>0 else float("nan")
    kBE_slice = (kH_min - eta_max*eta_max/kV_min) if kV_min>0 else float("nan")

    print("[SAFE-ball on nonlinear (functional) slice]")
    print(f"  beta={beta}, R={R}, nsamp={nsamp}, eps={eps}, root={root}")
    print(f"  kappa_H        = {kH_min}")
    print(f"  kappa_V        = {kV_min}")
    print(f"  eta            = {eta_max}")
    print(f"  kappa_BE_full  = {kBE_full}")
    print(f"  kappa_BE_slice = {kBE_slice}")
    print(f"  fp_min_eig     = {fp_min}")

    # identity sanity check
    X_id = np.zeros(36)
    kH0, kV0, eta0, kBE0_full, kBE0_slice, fp0 = constants_at_point(X_id, beta=beta, eps=eps, root=root)
    print("\n[At identity X=0]")
    print(f"  kappa_H        = {kH0}")
    print(f"  kappa_V        = {kV0}")
    print(f"  eta            = {eta0}")
    print(f"  kappa_BE_full  = {kBE0_full}")
    print(f"  kappa_BE_slice = {kBE0_slice}")
    print(f"  fp_min_eig     = {fp0}")

if __name__ == "__main__":
    sample_on_slice(beta=1.0, R=0.05, nsamp=10, eps=2e-5, seed=7, root=0)


[SAFE-ball on nonlinear (functional) slice]
  beta=1.0, R=0.05, nsamp=10, eps=2e-05, root=0
  kappa_H        = 1.5310615886675996
  kappa_V        = 0.6979596066091116
  eta            = 0.2958494602954765
  kappa_BE_full  = 0.6979596066091116
  kappa_BE_slice = 1.4056576221223784
  fp_min_eig     = 0.088238424208207

[At identity X=0]
  kappa_H        = 1.531247975618001
  kappa_V        = 0.6979128385882595
  eta            = 0.2956471857456481
  kappa_BE_full  = 0.6979128385882595
  kappa_BE_slice = 1.406007037772339
  fp_min_eig     = 0.08824243681815884
